# P9-2 · Train ở quy mô lớn

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Linear(16, 4); opt = torch.optim.SGD(model.parameters(), lr=0.1); loss_fn = nn.CrossEntropyLoss()
loader = [(torch.randn(8, 16), torch.randint(0, 4, (8,))) for _ in range(8)]     # 8 micro-batch, mỗi cái 8 mẫu

accum = 4
opt.zero_grad()
for i, (x, y) in enumerate(loader):
    loss = loss_fn(model(x), y) / accum          # chia số bước tích lũy
    loss.backward()                              # gradient cộng dồn vào .grad
    if (i + 1) % accum == 0:
        opt.step(); opt.zero_grad()
        print(f"cập nhật sau micro-batch {i + 1}, batch hiệu dụng {accum * 8}")

cập nhật sau micro-batch 4, batch hiệu dụng 32
cập nhật sau micro-batch 8, batch hiệu dụng 32


In [ ]:
from torch.utils.checkpoint import checkpoint

class Net(nn.Module):
    def __init__(self, n=6, d=256):
        super().__init__()
        self.blocks = nn.ModuleList(nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)) for _ in range(n))
    def forward(self, x):
        for block in self.blocks:
            x = checkpoint(block, x, use_reentrant=False)   # không lưu activation bên trong block
        return x
# với model transformers: model.gradient_checkpointing_enable()

net, x = Net(), torch.randn(64, 256, requires_grad=True)
net(x).sum().backward()                                    # backward tự tính lại forward từng block
print("gradient OK:", net.blocks[0][0].weight.grad.shape)

gradient OK: torch.Size([1024, 256])


## Bài tập 5: Train DDP {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần máy có 2 GPU: lưu thành train_ddp.py rồi chạy torchrun --nproc_per_node=2 train_ddp.py
import os, torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def main():
    dist.init_process_group("nccl")
    rank, local = dist.get_rank(), int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local)
    ds = TensorDataset(torch.randn(10000, 128), torch.randint(0, 10, (10000,)))
    sampler = DistributedSampler(ds, shuffle=True)
    dl = DataLoader(ds, batch_size=64, sampler=sampler)
    model = DDP(torch.nn.Sequential(torch.nn.Linear(128, 256), torch.nn.ReLU(), torch.nn.Linear(256, 10)).cuda(), device_ids=[local])
    opt = torch.optim.AdamW(model.parameters(), 1e-3)
    for epoch in range(3):
        sampler.set_epoch(epoch)                       # xáo khác nhau mỗi epoch
        for x, y in dl:
            x, y = x.cuda(), y.cuda()
            loss = torch.nn.functional.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()   # all-reduce gradient tự động
        if rank == 0:
            print(epoch, loss.item())
            torch.save(model.module.state_dict(), "ckpt.pt")
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
# chạy: torchrun --nproc_per_node=2 train_ddp.py